# Aula 02 - ML, IA e Ciência de Dados

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**07/08/2026 - Sprint 1 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula02.ipynb)

## O que este notebook é

É a **fase 2 do CRISP-DM**, Entendimento dos dados, executada. Não treina modelo nenhum: ele
produz a evidência dos seis pilares de qualidade sobre as cinco séries do case, para você não
ter que acreditar nos números do slide.

Continua **sem pandas**: pandas entra na Aula 03. Fazer na mão agora é o que torna visível o
que o pandas vai passar a fazer por você.

## Ao final deste notebook você terá

1. carregado as cinco séries do case de uma vez;
2. medido a **completude** de cada uma (quantos registros, de quando a quando);
3. verificado a **consistência** de unidade, e visto por que somar as cinco não faz sentido;
4. provado a **unicidade** de período em cada série;
5. medido a **atualidade** e o tamanho do salto até o horizonte pedido pelo parceiro;
6. validado o formato do período (**validade**) e a ordem de grandeza (**acurácia**);
7. encontrado, em código, a janela comum às cinco séries, que é o que o Modelo 1 vai poder usar.

## 1. Onde estão os arquivos

Mesma célula de resolução de caminho da Aula 01, agora para os cinco arquivos: funciona no
repositório clonado (onde os CSVs estão em `../dados/`) e no Colab (onde são baixados da versão
publicada do repositório).

In [ ]:
import csv
import os
import re
import urllib.request

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]

BASE_LOCAL = os.path.join("..", "dados")
BASE_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
              "main/dados/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(BASE_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            urllib.request.urlretrieve(BASE_BRUTA + arquivo, arquivo)
        caminhos[nome] = arquivo

for nome, caminho in caminhos.items():
    print("%-16s -> %s" % (nome, caminho))

## 2. Carregar as cinco séries

Um dicionário de séries, cada uma uma lista de tuplas `(periodo, valor, unidade)`. Dicionário e
tupla são as estruturas da Aula 01: a chave é o nome da série, e a tupla é um registro que não
deve ser alterado por acidente.

In [ ]:
def carregar(caminho):
    with open(caminho, encoding="utf-8") as arquivo:
        leitor = csv.reader(arquivo)
        next(leitor)                       # descarta o cabecalho
        return [(p, v, u) for p, v, u in leitor]

dados = {nome: carregar(caminho) for nome, caminho in caminhos.items()}

for nome, registros in dados.items():
    print("%-16s %3d registros" % (nome, len(registros)))

## 3. Completude

Quantos registros, e de quando a quando. É o pilar mais fácil de medir e o que mais surpreende
aqui.

In [ ]:
print("%-16s %5s  %-9s  %-9s" % ("serie", "n", "primeiro", "ultimo"))
for nome, registros in dados.items():
    print("%-16s %5d  %-9s  %-9s"
          % (nome, len(registros), registros[0][0], registros[-1][0]))

Quatro séries têm 117 registros, começando em `1997-T1`. A produção de ovos tem **157**,
começando em `1987-T1`: dez anos a mais.

Isso não é um erro do arquivo, é uma característica da fonte. Mas tem consequência direta de
modelagem: um alinhamento das cinco séries pelo período só pode usar a **interseção**, e os dez
anos extras de ovos ficam de fora. A célula da seção 8 calcula essa perda.

## 4. Consistência: as unidades

Cada arquivo tem unidade única (a Aula 01 provou isso para bovinos com um conjunto). A pergunta
agora é outra: as cinco séries têm a **mesma** unidade entre si?

In [ ]:
unidades = {nome: {u for _, _, u in registros}
            for nome, registros in dados.items()}

for nome, us in unidades.items():
    assert len(us) == 1, "%s tem mais de uma unidade" % nome
    print("%-16s %s" % (nome, list(us)[0]))

distintas = {list(us)[0] for us in unidades.values()}
print()
print("unidades distintas entre as cinco series:", len(distintas))
print(sorted(distintas))

### O número que o Python calcula e que não significa nada

A célula abaixo soma o último trimestre das cinco séries. **Ela não vai dar erro.** Vai devolver
um número, e esse número é lixo: soma quilogramas com mil litros e mil dúzias.

É o oposto do `KeyError` da Aula 01. Lá, o Python interrompeu e apontou o problema. Aqui ele
colabora com o erro, e nada na saída avisa que a conta não tem sentido físico. É por isso que
consistência de unidade é assunto de aula e não de validador.

In [ ]:
soma_sem_sentido = sum(float(registros[-1][1])
                       for registros in dados.values())

print("soma do ultimo trimestre das cinco series:", soma_sem_sentido)
print()
print("o que foi somado:")
for nome, registros in dados.items():
    print("  %-16s %18.1f  %s" % (nome, float(registros[-1][1]), registros[-1][2]))

## 5. Unicidade

Cada período pode aparecer no máximo uma vez por série. Este é o pilar que `tools/baixar_dados.py`
protege ao filtrar as dimensões extras das tabelas SIDRA (tipo de rebanho, tipo de inspeção, mês
dentro do trimestre) pelo valor "Total".

Sem esse filtro, o mesmo trimestre voltaria várias vezes com recortes diferentes, e qualquer soma
ficaria inflada sem nenhum sinal de erro.

In [ ]:
for nome, registros in dados.items():
    periodos = [p for p, _, _ in registros]
    repetidos = {p for p in periodos if periodos.count(p) > 1}
    print("%-16s %3d periodos, %3d distintos, repetidos: %s"
          % (nome, len(periodos), len(set(periodos)), sorted(repetidos) or "nenhum"))
    assert not repetidos, "%s tem periodo repetido" % nome

## 6. Atualidade

O dado mais recente é `2026-T1`. O parceiro quer previsão para 8 trimestres à frente. Quanto
tempo de 2026 já existe na base, e para onde o horizonte aponta?

In [ ]:
def proximos_trimestres(ultimo, quantos):
    ano, tri = int(ultimo[:4]), int(ultimo[-1])
    saida = []
    for _ in range(quantos):
        tri += 1
        if tri == 5:
            tri, ano = 1, ano + 1
        saida.append("%d-T%d" % (ano, tri))
    return saida

ultimo = dados["abate_bovinos"][-1][0]
de_2026 = [p for p, _, _ in dados["abate_bovinos"] if p.startswith("2026")]

print("ultimo trimestre medido:", ultimo)
print("trimestres de 2026 na base:", de_2026, "(de 4 possiveis)")
print()
print("horizonte de 8 trimestres pedido pelo parceiro:")
print(" ", proximos_trimestres(ultimo, 8))

Oito trimestres à frente de `2026-T1` chegam a `2028-T1`: os mesmos 24 meses que o TAPI pediu,
na granularidade que a fonte permite medir.

E repare no dado parcial: 2026 tem **1 de 4** trimestres. Comparar o total de 2026 com o de um ano
completo é comparar um trimestre com quatro, erro que não levanta exceção nenhuma.

## 7. Validade e acurácia

**Validade:** todo período precisa casar com o formato `AAAA-TN` do contrato. Se um valor
`202504` escapasse para cá, ele seria lido como abril de 2025 em vez do quarto trimestre.

**Acurácia:** a ordem de grandeza precisa ser compatível com o que a série mede. Valor na casa
das centenas seria sinal de ter pego "número de informantes" em vez da série física, que é o
risco de pedir `v/all` à API do SIDRA.

In [ ]:
FORMATO = re.compile(r"^\d{4}-T[1-4]$")

for nome, registros in dados.items():
    fora = [p for p, _, _ in registros if not FORMATO.match(p)]
    assert not fora, "%s tem periodo fora do contrato: %s" % (nome, fora[:3])
print("validade: todos os periodos das cinco series casam com AAAA-TN")
print()

print("%-16s %18s %18s  %s" % ("serie", "minimo", "maximo", "unidade"))
for nome, registros in dados.items():
    valores = [float(v) for _, v, _ in registros if v]
    print("%-16s %18.0f %18.0f  %s"
          % (nome, min(valores), max(valores), registros[0][2]))
    assert min(valores) > 10_000, "%s: ordem de grandeza suspeita" % nome

## 8. A janela comum às cinco séries

Aqui a completude deixa de ser observação e passa a ser decisão de modelagem: o Modelo 1 alinha as
cinco séries pelo período, então ele só pode usar a interseção.

In [ ]:
conjuntos = [{p for p, _, _ in registros} for registros in dados.values()]
comum = sorted(set.intersection(*conjuntos))

print("janela comum: %s a %s  (%d trimestres)"
      % (comum[0], comum[-1], len(comum)))
print()
for nome, registros in dados.items():
    perdidos = len(registros) - len(comum)
    print("%-16s %3d registros, %3d fora da janela comum" % (nome, len(registros), perdidos))

## 9. O risco que nenhuma célula acima pega

Todas as verificações deste notebook passaram. Os cinco arquivos estão íntegros, e continuam
íntegros depois de tudo o que foi medido.

E ainda assim há um problema sério na base, invisível para todas elas: a série de leite vem da
variável 282 da tabela 1086, **"Quantidade de leite cru, resfriado ou não, adquirido"**. Ela não
mede a produção de leite do Brasil. Mede o volume que os laticínios *compraram*.

Nenhum teste de formato, unicidade, unidade ou ordem de grandeza pode detectar isso, porque o
arquivo está correto. O que está errado é a interpretação de quem usar essa série como se fosse
produção total.

Para o Modelo 2 (produção de proteína virando demanda de ração), usar captação formal como proxy
de produção **subestima o rebanho leiteiro**, e esse erro entra como viés, não como ruído: ele não
desaparece com mais dado.

> A moral do notebook: a fase 2 do CRISP-DM não é um script de carga com asserts. É ler a
> documentação da fonte.

## 10. Desafio

Responda no código, e o item 3 em texto.

1. Se o Modelo 1 usar a janela comum, quantos **anos completos** ele tem para treinar?
2. A série de ovos está em "mil dúzias". Quantos ovos, em unidades, foram produzidos no trimestre
   de maior valor da série? (uma dúzia são 12 ovos, e a unidade já vem multiplicada por mil)
3. Dos seis pilares de qualidade, qual você considera o mais arriscado para o **Modelo 2**, e por
   quê? Escreva em `resposta_3`. Esta é a frase que entra na sua ART.1.

In [ ]:
# 1. anos completos dentro da janela comum
anos = {}
for p in comum:
    anos[p[:4]] = anos.get(p[:4], 0) + 1
completos = sorted(ano for ano, n in anos.items() if n == 4)
print("anos completos na janela comum:", len(completos),
      "(de %s a %s)" % (completos[0], completos[-1]))
print("anos parciais:", sorted(ano for ano, n in anos.items() if n != 4))

# 2. o maior trimestre de ovos, convertido de "mil duzias" para unidades
ovos = [(p, float(v)) for p, v, _ in dados["producao_ovos"] if v]
periodo_pico, mil_duzias = max(ovos, key=lambda par: par[1])
unidades_de_ovo = mil_duzias * 1_000 * 12
print()
print("pico de ovos: %s com %.0f mil duzias" % (periodo_pico, mil_duzias))
print("em unidades:  %.0f ovos (%.1f bilhoes)"
      % (unidades_de_ovo, unidades_de_ovo / 1_000_000_000))

# 3. escreva a sua resposta aqui, em uma ou duas frases
resposta_3 = ""
print()
print("resposta_3:", repr(resposta_3))